# Instalando Bibliotecas
No terminal, na pasta raiz:

``` bash
pip install -r requirements.txt

```

Após instalar todas as dependências necessária, selecione o Kernel equivalente ao .venv

# Importando Bibliotecas

In [2]:
import os
import shutil
import yaml
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
import cv2


# Modelo Geral

## Organizando o ambiente

In [11]:
# ==================== CONFIGURAÇÕES ====================
DATASET_ORIGINAL = "./Dataset_v2" 
DATASET_FINAL = "./animais_geral"

TREINO_PROP = 0.70
VAL_PROP = 0.20
TESTE_PROP = 0.10

# Pastas que contêm subpastas de espécies (não têm fotos direto nelas)
PASTAS_ESPECIALISTAS = ["Cobras", "Aranhas"]
# =======================================================

def criar_estrutura_pastas(base_path):
    for split in ['train', 'val', 'test']:
        if not os.path.exists(os.path.join(base_path, split)):
            os.makedirs(os.path.join(base_path, split))

def mapear_imagens():
    mapeamento = {}
    pastas_principais = [f for f in os.listdir(DATASET_ORIGINAL) if os.path.isdir(os.path.join(DATASET_ORIGINAL, f))]
    
    for pasta in pastas_principais:
        if pasta.startswith('.') or pasta == 'animais_geral' or pasta == '.venv':
            continue
            
        caminho_pasta = os.path.join(DATASET_ORIGINAL, pasta)
        
        # Define o nome da classe limpando o nome científico entre parênteses
        classe_geral = pasta.split('(')[0].strip()
        
        # Remove o plural de Cobras e Aranhas para ficar um nome mais limpo no modelo
        if classe_geral == "Cobras": classe_geral = "Cobra"
        if classe_geral == "Aranhas": classe_geral = "Aranha"
        
        # Caso Especial: Cobras e Aranhas
        if pasta in PASTAS_ESPECIALISTAS:
            subpastas = [s for s in os.listdir(caminho_pasta) if os.path.isdir(os.path.join(caminho_pasta, s))]
            
            for sub in subpastas:
                caminho_sub = os.path.join(caminho_pasta, sub)
                especie_nome = sub.split('(')[0].strip()
                
                for img in os.listdir(caminho_sub):
                    if img.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                        caminho_completo_img = os.path.join(caminho_sub, img)
                        novo_nome = f"{classe_geral}_{especie_nome}_{img}".replace(" ", "_")
                        
                        if classe_geral not in mapeamento:
                            mapeamento[classe_geral] = []
                        mapeamento[classe_geral].append((caminho_completo_img, novo_nome))
        
        # Caso Normal: Outros animais
        else:
            for img in os.listdir(caminho_pasta):
                if img.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                    caminho_completo_img = os.path.join(caminho_pasta, img)
                    novo_nome = f"{classe_geral}_{img}".replace(" ", "_")
                    
                    if classe_geral not in mapeamento:
                        mapeamento[classe_geral] = []
                    mapeamento[classe_geral].append((caminho_completo_img, novo_nome))
                    
    return mapeamento

def gerar_yaml(classes_detectadas):
    """Gera automaticamente o arquivo data.yaml estruturado para o YOLO-cls."""
    # O YOLO classifica em ordem alfabética com base nas pastas existentes
    classes_ordenadas = sorted(list(classes_detectadas))
    
    dict_yaml = {
        'path': os.path.abspath(DATASET_FINAL), # Caminho absoluto para evitar erros do YOLO
        'train': 'train',
        'val': 'val',
        'test': 'test',
        'nc': len(classes_ordenadas),
        'names': {i: classe for i, classe in enumerate(classes_ordenadas)}
    }
    
    caminho_yaml = os.path.join(DATASET_FINAL, "data.yaml")
    
    # Gravando o arquivo YAML de forma limpa
    with open(caminho_yaml, 'w', encoding='utf-8') as f:
        yaml.dump(dict_yaml, f, allow_unicode=True, default_flow_style=False, sort_keys=False)
        
    print(f" Arquivo YAML gerado com sucesso em: {caminho_yaml}")

def processar_e_dividir():
    criar_estrutura_pastas(DATASET_FINAL)
    dataset_mapeado = mapear_imagens()
    
    classes_processadas = set()
    
    for classe, lista_dados in dataset_mapeado.items():
        total_imagens = len(lista_dados)
        print(f"Processando '{classe}': {total_imagens} imagens encontradas.")
        
        if total_imagens < 3:
            print(f" Imagens insuficientes para dividir a classe {classe}. Pulando...")
            continue
            
        classes_processadas.add(classe)
        
        indices = list(range(total_imagens))
        prop_restante = VAL_PROP + TESTE_PROP
        
        idx_treino, idx_restante = train_test_split(indices, test_size=prop_restante, random_state=42)
        
        prop_val_relativa = VAL_PROP / prop_restante
        idx_val, idx_teste = train_test_split(idx_restante, test_size=(1.0 - prop_val_relativa), random_state=42)
        
        divisoes = {'train': idx_treino, 'val': idx_val, 'test': idx_teste}
        
        for split, indices_split in divisoes.items():
            pasta_destino_final = os.path.join(DATASET_FINAL, split, classe)
            os.makedirs(pasta_destino_final, exist_ok=True)
            
            for idx in indices_split:
                caminho_origem, novo_nome_arquivo = lista_dados[idx]
                caminho_destino = os.path.join(pasta_destino_final, novo_nome_arquivo)
                shutil.copy2(caminho_origem, caminho_destino)
                
    # Cria o arquivo .yaml com as classes que realmente foram processadas
    if classes_processadas:
        gerar_yaml(classes_processadas)
        
    print("\n Tudo pronto!")

if __name__ == "__main__":
    processar_e_dividir() 

Processando 'Iguana-Verde': 15 imagens encontradas.
Processando 'Sapo-Cururu': 15 imagens encontradas.
Processando 'Preguiça-de-Bentinho': 25 imagens encontradas.
Processando 'Cobra': 21 imagens encontradas.
Processando 'Calango-de-Pedra': 20 imagens encontradas.
Processando 'Macaco-de-Cheiro': 30 imagens encontradas.
Processando 'Cutia-de-Crista': 40 imagens encontradas.
Processando 'Jacaretinga': 10 imagens encontradas.
Processando 'Aranha': 67 imagens encontradas.
Processando 'Parauacu-de-Cara-Branca': 25 imagens encontradas.
 Arquivo YAML gerado com sucesso em: ./animais_geral/data.yaml

 Tudo pronto!


## Ajuste fino do modelo geral

In [12]:

def treinar_modelo_geral():
    # Carrega o modelo pré-treinado Nano. Usando a versão '-cls' que é específica para classificação pura
    model = YOLO('yolo26n-cls.pt')

    # Inicia o treinamento com os parâmetros ajustados para datasets pequenos
    results = model.train(
        data='./animais_geral',    # Caminho da pasta que o script anterior criou
        epochs=100,                # 100 épocas 
        imgsz=224,                 # Resolução padrão e leve para classificação
        batch=8,                   # Batch menor (8 ou 16) ajuda a estabilizar datasets pequenos
        workers=4,                 # Linhas de processamento para carregar fotos
        device=0,                  # Use 0 para GPU Nvidia, ou 'cpu' se não tiver placa de vídeo
        
        # ================= HIPERPARAMETROS DE REGULARIZAÇÃO =================
        lr0=0.005,                 # Taxa de aprendizado inicial baixa para não destruir os pesos pré-treinados
        lrf=0.01,                  # Taxa de aprendizado final
        weight_decay=0.001,        # Penaliza pesos muito grandes para evitar Overfitting
        dropout=0.2,               # Desativa 20% dos neurônios aleatoriamente no treino para forçar o modelo a aprender geral
        
        # ================= DATA AUGMENTATION (AUMENTO DE DADOS) =================
        degrees=30.0,              # Rotaciona as fotos aleatoriamente em até 30 graus (muda o ângulo)
        scale=0.5,                 # Zoom aleatório (reduz ou aumenta a foto para simular distância)
        translate=0.1,             # Desloca o animal um pouco para os lados/cima/baixo
        perspective=0.0001,        # Pequena alteração de perspectiva 3D
        flipud=0.0,                # Não vira de ponta cabeça (geralmente animais não andam de cabeça para baixo)
        fliplr=0.5,                # Espelha a foto horizontalmente (50% de chance) - ótimo para mudar o lado do animal
        bgr=0.1,                   # Altera levemente as cores (simula dias mais frios/quentes ou iluminação)

        project='Triagem',  # Nome da pasta principal
        name='Geral'          # Nome da subpasta atual
    )
    
    print("Treinamento concluído! Os pesos foram salvos na pasta 'runs/classify/train'")

if __name__ == "__main__":
    treinar_modelo_geral()

Ultralytics 8.4.60 🚀 Python-3.13.11 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce GTX 1650, 3897MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.1, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./animais_geral, degrees=30.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.2, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=Geral, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10

## Testando a classificação


In [29]:
caminho_imagem = "imagens_teste/cobra_escada2.jpg"

In [30]:
def testar_previsao(caminho_imagem):  #
    # 1. Caminho para o modelo treinado
    caminho_modelo = './runs/classify/train/weights/best.pt' 
    
    if not os.path.exists(caminho_modelo):
        print(f" Erro: O arquivo do modelo não foi encontrado em '{caminho_modelo}'.")
        return

    # 2. Carrega o modelo
    model = YOLO(caminho_modelo)

    if not os.path.exists(caminho_imagem):
        print(f" Imagem inválida ou não encontrada em '{caminho_imagem}'!")
        return

    # 3. Executa a previsão
    resultados = model.predict(source=caminho_imagem, verbose=False)

    # 4. Processa o resultado
    for resultado in resultados:
        nomes_classes = resultado.names
        probs = resultado.probs

        # Palpite principal (Top 1)
        id_classe_top1 = probs.top1
        nome_animal_top1 = nomes_classes[id_classe_top1]
        confianca_top1 = probs.top1conf.item() * 100

        print("\n" + "="*40)
        print(f" RESULTADO DA ANÁLISE:")
        print(f"Animal identificado: {nome_animal_top1}")
        print(f"Certeza do modelo: {confianca_top1:.2f}%")
        print("="*40)

        print("\n Outros palpites do modelo:")
        top5_indices = probs.top5
        
        # Mostra até os 3 primeiros palpites que o modelo teve
        for i in range(min(3, len(top5_indices))): 
            idx = top5_indices[i]
            nome = nomes_classes[idx]
            conf = probs.data[idx].item() * 100
            print(f"   - {nome}: {conf:.2f}%")

if __name__ == "__main__":
    testar_previsao(caminho_imagem)


 RESULTADO DA ANÁLISE:
Animal identificado: Cobra
Certeza do modelo: 99.93%

 Outros palpites do modelo:
   - Cobra: 99.93%
   - Aranha: 0.06%
   - Sapo-Cururu: 0.00%


# Modelos especialistas: Aranhas

## Organizando Ambiente

In [5]:

ALVO = "Aranhas" 

DATASET_ORIGINAL = f"./Dataset_v2/{ALVO}" 
DATASET_FINAL = f"./animais_especialista_{ALVO.lower()}"

TREINO_PROP = 0.70
VAL_PROP = 0.20
TESTE_PROP = 0.10
# =======================================================

def criar_estrutura_pastas(base_path):
    for split in ['train', 'val', 'test']:
        os.makedirs(os.path.join(base_path, split), exist_ok=True)

def processar_especialista():
    criar_estrutura_pastas(DATASET_FINAL)
    
    # Lista as espécies
    especies = [f for f in os.listdir(DATASET_ORIGINAL) if os.path.isdir(os.path.join(DATASET_ORIGINAL, f))]
    classes_processadas = set()
    
    for especie in especies:
        pasta_especie = os.path.join(DATASET_ORIGINAL, especie)
        imagens = [img for img in os.listdir(pasta_especie) if img.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))]
        
        if len(imagens) < 3:
            print(f" Poucas imagens para {especie}. Pulando...")
            continue
            
        classes_processadas.add(especie)
        print(f"Processando espécie '{especie}': {len(imagens)} imagens encontradas.")
        
        # Divisão dos dados
        indices = list(range(len(imagens)))
        prop_restante = VAL_PROP + TESTE_PROP
        idx_treino, idx_restante = train_test_split(indices, test_size=prop_restante, random_state=42)
        
        prop_val_relativa = VAL_PROP / prop_restante
        idx_val, idx_teste = train_test_split(idx_restante, test_size=(1.0 - prop_val_relativa), random_state=42)
        
        divisoes = {'train': idx_treino, 'val': idx_val, 'test': idx_teste}
        
        for split, indices_split in divisoes.items():
            pasta_destino = os.path.join(DATASET_FINAL, split, especie)
            os.makedirs(pasta_destino, exist_ok=True)
            
            for idx in indices_split:
                img = imagens[idx]
                shutil.copy2(os.path.join(pasta_especie, img), os.path.join(pasta_destino, f"{especie}_{img}"))
                
    # Gerar o YAML específico do especialista
    classes_ordenadas = sorted(list(classes_processadas))
    dict_yaml = {
        'path': os.path.abspath(DATASET_FINAL),
        'train': 'train', 'val': 'val', 'test': 'test',
        'nc': len(classes_ordenadas),
        'names': {i: c for i, c in enumerate(classes_ordenadas)}
    }
    
    with open(os.path.join(DATASET_FINAL, "data.yaml"), 'w', encoding='utf-8') as f:
        yaml.dump(dict_yaml, f, allow_unicode=True, default_flow_style=False, sort_keys=False)
        
    print(f" Dataset especialista de {ALVO} pronto em '{DATASET_FINAL}'!")

if __name__ == "__main__":
    processar_especialista()

Processando espécie 'Armadeira-da-Bananeira ( Phoneutria Phera)': 14 imagens encontradas.
Processando espécie 'Armadeira (Phoneutria Reidyi)': 17 imagens encontradas.
Processando espécie 'Tarântula (Megaphobema velvetosoma)': 10 imagens encontradas.
Processando espécie 'Aranha-de-Prata (Argiope argentata)': 12 imagens encontradas.
Processando espécie 'Tarântula (Avicularia juruensis)': 14 imagens encontradas.
 Dataset especialista de Aranhas pronto em './animais_especialista_aranhas'!


## Ajuste fino do modelo especialista de aranhas

In [7]:
def treinar_modelo_especialista_aranhas():
    model = YOLO('yolo26n-cls.pt') 

    # Inicia o treino apontando para o dataset do especialista
    results = model.train(
        data='./animais_especialista_aranhas/',
        epochs=120,                
        imgsz=224,                 
        batch=8,                   
        workers=4,                 
        device='0',              
        
        # ================= HIPERPARAMETROS AJUSTADOS PARA ESPECIALISTAS =================
        lr0=0.002,                 # Taxa de aprendizado ainda menor (ajuste fino cirúrgico)
        lrf=0.01,                  
        weight_decay=0.002,        # Regularização um pouco maior para não decorar detalhes do fundo
        dropout=0.3,               # 30% para forçar o modelo a focar no animal, não no cenário
        
        # ================= DATA AUGMENTATION INTENSO =================
        degrees=45.0,              # Rotação maior (aranhas andam em paredes e tetos, mudando muito o ângulo)
        scale=0.5,                 
        translate=0.15,            
        perspective=0.0001,        
        flipud=0.5,                # Aranhas podem aparecer de cabeça para baixo em galhos/folhas
        fliplr=0.5,                
        bgr=0.15,                  # Simula variações na iluminação (flash da câmera, luz do sol, sombra)

        project='Triagem',  # Nome da pasta principal
        name='Especialista_Aranhas'          # Nome da subpasta atual
    )
    
    print("Treinamento do modelo Especialista em Aranhas concluído!")

if __name__ == "__main__":
    treinar_modelo_especialista_aranhas()

Ultralytics 8.4.60 🚀 Python-3.13.11 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce GTX 1650, 3897MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.15, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./animais_especialista_aranhas/, degrees=45.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.3, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=Especialista_Aranhas, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto

# Modelos especialistas: Cobras

## Organizando Ambiente

In [9]:

ALVO = "Cobras" 

DATASET_ORIGINAL = f"./Dataset_v2/{ALVO}" 
DATASET_FINAL = f"./animais_especialista_{ALVO.lower()}"

TREINO_PROP = 0.70
VAL_PROP = 0.20
TESTE_PROP = 0.10
# =======================================================

def criar_estrutura_pastas(base_path):
    for split in ['train', 'val', 'test']:
        os.makedirs(os.path.join(base_path, split), exist_ok=True)

def processar_especialista():
    criar_estrutura_pastas(DATASET_FINAL)
    
    # Lista as espécies
    especies = [f for f in os.listdir(DATASET_ORIGINAL) if os.path.isdir(os.path.join(DATASET_ORIGINAL, f))]
    classes_processadas = set()
    
    for especie in especies:
        pasta_especie = os.path.join(DATASET_ORIGINAL, especie)
        imagens = [img for img in os.listdir(pasta_especie) if img.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))]
        
        if len(imagens) < 3:
            print(f" Poucas imagens para {especie}. Pulando...")
            continue
            
        classes_processadas.add(especie)
        print(f"Processando espécie '{especie}': {len(imagens)} imagens encontradas.")
        
        # Divisão dos dados
        indices = list(range(len(imagens)))
        prop_restante = VAL_PROP + TESTE_PROP
        idx_treino, idx_restante = train_test_split(indices, test_size=prop_restante, random_state=42)
        
        prop_val_relativa = VAL_PROP / prop_restante
        idx_val, idx_teste = train_test_split(idx_restante, test_size=(1.0 - prop_val_relativa), random_state=42)
        
        divisoes = {'train': idx_treino, 'val': idx_val, 'test': idx_teste}
        
        for split, indices_split in divisoes.items():
            pasta_destino = os.path.join(DATASET_FINAL, split, especie)
            os.makedirs(pasta_destino, exist_ok=True)
            
            for idx in indices_split:
                img = imagens[idx]
                shutil.copy2(os.path.join(pasta_especie, img), os.path.join(pasta_destino, f"{especie}_{img}"))
                
    # Gerar o YAML específico do especialista
    classes_ordenadas = sorted(list(classes_processadas))
    dict_yaml = {
        'path': os.path.abspath(DATASET_FINAL),
        'train': 'train', 'val': 'val', 'test': 'test',
        'nc': len(classes_ordenadas),
        'names': {i: c for i, c in enumerate(classes_ordenadas)}
    }
    
    with open(os.path.join(DATASET_FINAL, "data.yaml"), 'w', encoding='utf-8') as f:
        yaml.dump(dict_yaml, f, allow_unicode=True, default_flow_style=False, sort_keys=False)
        
    print(f" Dataset especialista de {ALVO} pronto em '{DATASET_FINAL}'!")

if __name__ == "__main__":
    processar_especialista()

Processando espécie 'Jiboia (Boa constrictor)': 10 imagens encontradas.
Processando espécie 'Jararaca-do-Norte (Bothrops atrox)': 11 imagens encontradas.
 Dataset especialista de Cobras pronto em './animais_especialista_cobras'!


In [10]:

def treinar_modelo_especialista_cobras():
    model = YOLO('yolo26n-cls.pt') 

    # Inicia o treino apontando para o dataset do especialista em cobras
    results = model.train(
        data='./animais_especialista_cobras', 
        epochs=120,                
        imgsz=224,                 
        batch=8,                   
        workers=4,                 
        device='0',        
        
        # ================= HIPERPARAMETROS AJUSTADOS PARA ESPECIALISTAS =================
        lr0=0.002,                 # Taxa de aprendizado baixa para ajuste fino cirúrgico
        lrf=0.01,                  
        weight_decay=0.003,        # Aumentado levemente para evitar que o modelo decore o chão/folhas da camuflagem
        dropout=0.3,               # Desativa 30% dos neurônios para focar nos padrões de escamas e cabeça
        
        # ================= DATA AUGMENTATION AJUSTADO PARA COBRAS =================
        degrees=90.0,              # Aumentado para 90°: cobras podem ser fotografadas em qualquer direção/ângulo no chão
        scale=0.6,                 # Zoom flexível (simula cobras mais de perto ou mais distantes/escondidas)
        translate=0.2,             # Aumentado para 0.2: ajuda o modelo a focar na cobra mesmo se ela estiver em um canto da foto
        perspective=0.0005,        # Aumentado levemente: simula fotos tiradas de cima (em pé) ou rasteiras (nível do chão)
        flipud=0.5,                # Cobras mudam de sentido vertical facilmente ao subirem em galhos
        fliplr=0.5,                # Inverte horizontalmente (muda o lado para onde a cobra está rastejando)
        bgr=0.15,                  # Altera tons de cores (ajuda com iluminações de mata fechada ou sol direto)

        project='Triagem',  # Nome da pasta principal
        name='Especialista_Cobras'          # Nome da subpasta atual
    )
    
    print("Treinamento do especialista em Cobras concluído!")

if __name__ == "__main__":
    treinar_modelo_especialista_cobras()

Ultralytics 8.4.60 🚀 Python-3.13.11 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce GTX 1650, 3897MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.15, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./animais_especialista_cobras, degrees=90.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.3, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=Especialista_Cobras, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, o

# Fluxo da Triagem 

In [20]:
def executar_pipeline_fauna(caminho_imagem, 
                            MODELO_GERAL_PATH = './runs/classify/Triagem/Geral/weights/best.pt',
                            MODELO_ARANHA_PATH = './runs/classify/Triagem/Especialista_Aranhas/weights/best.pt',
                            MODELO_COBRA_PATH = './runs/classify/Triagem/Especialista_Cobras/weights/best.pt'
):
    # Verificando se o caminho da imagem é válido
    if not os.path.exists(caminho_imagem):
        print(f" Erro: Imagem inválida no caminho '{caminho_imagem}'")
        return

    # Validando se os modelos existem antes de começar

    # Modelo Geral
    elif not os.path.exists(MODELO_GERAL_PATH):
        print(f" Erro: Modelo geral não encontrado em '{MODELO_GERAL_PATH}'")
        return
    
    # Modelo Especialista em Aranhas
    elif not os.path.exists(MODELO_ARANHA_PATH):
        print(f" Erro: Modelo Especialista em Aranhas não encontrado em '{MODELO_ARANHA_PATH}'")
        return

    # Modelo Especialista em Cobras
    elif not os.path.exists(MODELO_COBRA_PATH):
        print(f" Erro: Modelo Especialista em Cobras não encontrado em '{MODELO_COBRA_PATH}'")
        return

    # Carrega o Modelo Geral
    modelo_geral = YOLO(MODELO_GERAL_PATH)

    # 3. Executa a primeira classificação (Modelo Geral)
    print(f" Caminho da imagem: {caminho_imagem}")
    pred_geral = modelo_geral.predict(source=caminho_imagem, verbose=False)[0]
    
    # Extrai o nome do animal e a confiança do modelo geral
    classe_geral = pred_geral.names[pred_geral.probs.top1]
    confianca_geral = pred_geral.probs.top1conf.item() * 100

    print(f" [Modelo Geral]: Detectado '{classe_geral}' com {confianca_geral:.2f}% de certeza.")

    # Definimos um limite mínimo de confiança para evitar acionar especialistas com palpites errados
    LIMITE_CONFIANCA = 60.0 
    
    # O modelo geral disse que é uma Aranha
    if classe_geral == "Aranha" and confianca_geral >= LIMITE_CONFIANCA:
        print(" Acionando Modelo Especialista em Aranhas")
        
        if os.path.exists(MODELO_ARANHA_PATH):
            modelo_aranha = YOLO(MODELO_ARANHA_PATH)
            pred_especialista = modelo_aranha.predict(source=caminho_imagem, verbose=False)[0]
            
            especie = pred_especialista.names[pred_especialista.probs.top1]
            conf_especie = pred_especialista.probs.top1conf.item() * 100
            
            print(f"\nRESULTADO FINAL: {especie} ({conf_especie:.2f}% de certeza)")
        else:
            print("Erro: Arquivo do modelo especialista em Aranhas não foi encontrado.")

    # O modelo geral disse que é uma Cobra
    elif classe_geral == "Cobra" and confianca_geral >= LIMITE_CONFIANCA:
        print("Acionando Modelo Especialista em Cobras...")
        
        if os.path.exists(MODELO_COBRA_PATH):
            modelo_cobra = YOLO(MODELO_COBRA_PATH)
            pred_especialista = modelo_cobra.predict(source=caminho_imagem, verbose=False)[0]
            
            especie = pred_especialista.names[pred_especialista.probs.top1]
            conf_especie = pred_especialista.probs.top1conf.item() * 100
            
            print(f"\nRESULTADO FINAL: {especie} ({conf_especie:.2f}% de certeza)")
        else:
            print("Erro: Arquivo do modelo especialista em Cobras não foi encontrado.")

    # Caso 3: Qualquer outro animal (Iguana, Sapo, Cutia.)
    else:
        if confianca_geral < LIMITE_CONFIANCA:
            print("\nA confiança do modelo foi muito baixa para dar um veredito seguro.")
            print(f"Palpite incerto: {classe_geral} ({confianca_geral:.2f}%)")
        else:
            print(f"\nRESULTADO FINAL: {classe_geral} ({confianca_geral:.2f}% de certeza)")

In [21]:
imagem = "imagens_teste/Avicularia_juruensis_female_morphotype_2_ZK119.jpg"
executar_pipeline_fauna(imagem)

 Caminho da imagem: imagens_teste/Avicularia_juruensis_female_morphotype_2_ZK119.jpg
 [Modelo Geral]: Detectado 'Aranha' com 99.57% de certeza.
 Acionando Modelo Especialista em Aranhas

RESULTADO FINAL: Tarântula (Avicularia juruensis) (99.79% de certeza)
